# Setup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
#algorithms
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

#metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

#AIF360 metrics
from aif360.metrics import ClassificationMetric
from aif360.datasets import BinaryLabelDataset

#auxiliar
from pathlib import Path
import sys
import os
auxiliar_path = os.path.join(os.getcwd(), '..', 'auxiliar')
sys.path.append(auxiliar_path)
import config

c:\Users\gabri\anaconda3\envs\fairness-research\Lib\site-packages\inFairness\utils\ndcg.py:37: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  vect_normalized_discounted_cumulative_gain = vmap(
c:\Users\gabri\anaconda3\envs\fairness-research\Lib\site-packages\inFairness\utils\ndcg.py:48: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.vmap` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.vmap` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  mon

In [3]:
base_path = Path.cwd().parent / "data" / "raw"
destination_path = Path.cwd().parent / "data" / "processed"

Variáveis

# Arrhythmia

In [55]:
# exp_config = config.DATASET_CONFIGS["DIABETES"]

# seed = config.SEED
# target_name = config.TARGET_NAME
# sen_var = exp_config["sen_var"]
# dataset_code = exp_config["dataset_code"]

dataset_code = "11_arrhythmia"  # exp_config["dataset_code"]

In [56]:
from pathlib import Path
import pandas as pd

base_path = Path("../data/raw").resolve()
datapath = base_path / dataset_code

df = pd.read_csv(datapath / "arrhythmia.data", header=None)


In [57]:
df.shape

(452, 280)

In [58]:
target_raw = df.iloc[:, -1]

df["target"] = target_raw.apply(lambda x: 0 if x == 1 else 1)

# (Opcional) Remover a coluna original
df.drop(columns=df.columns[-2], inplace=True)  # última coluna antes da nova "target"

# Verifica distribuição
print(df["target"].value_counts())

target
0    245
1    207
Name: count, dtype: int64


In [59]:
for col in df.columns:
    n_q = (df[col] == "?").sum()
    if n_q > 0:
        print(f"Coluna '{col}' possui {n_q} valores '?'")


Coluna '10' possui 8 valores '?'
Coluna '11' possui 22 valores '?'
Coluna '12' possui 1 valores '?'
Coluna '13' possui 376 valores '?'
Coluna '14' possui 1 valores '?'


In [63]:
(452-376)/452

0.168141592920354

In [ ]:

# Coluna a excluir totalmente
cols_to_drop = [13]
df = df.drop(columns=cols_to_drop)

# Colunas a manter e imputar valores "?"
cols_to_fill = [10, 11, 12, 14]

# Substitui "?" por NaN
df[cols_to_fill] = df[cols_to_fill].replace("?", np.nan)

# Converte para float
df[cols_to_fill] = df[cols_to_fill].astype(float)

# Imputa a mediana 
for col in cols_to_fill:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

df = df.reset_index(drop=True)


In [61]:
sex_col = 1  


df[sex_col] = pd.to_numeric(df[sex_col], errors="coerce")


df[sex_col] = df[sex_col].map({0: 1, 1: 0})


df.rename(columns={sex_col: "sex"}, inplace=True)

print(df["sex"].value_counts())


sex
0    249
1    203
Name: count, dtype: int64


In [62]:
base_path = Path("../data/processed")  
datapath = base_path / dataset_code  

df.to_csv(datapath / f"{dataset_code}_pp.csv", index=False)

# Mental

In [28]:
df = pd.read_csv(base_path / f"15_mental/mental.csv")

In [23]:
df.shape[0]

1259

In [24]:
df.isna().sum()

Timestamp                       0
Age                             0
Gender                          0
Country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1095
dtype: int64

In [29]:
df.drop(columns=['Timestamp','state','comments'], inplace=True)

In [30]:
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.lower())

In [31]:
df.rename(columns={"treatment": "target"}, inplace=True)

In [32]:
df.dropna(subset=["self_employed"], inplace=True)

In [ ]:
df['work_interfere'] = df['work_interfere'].fillna('Missing')

dummies = pd.get_dummies(df['work_interfere'], prefix='work_interfere')

df = pd.concat([df, dummies], axis=1)
df = df.drop(columns=['work_interfere'], inplace=True)

In [36]:
cols_yes_no = [
    'self_employed', 'family_history', 'target', 'remote_work', 'tech_company',
    'benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'mental_health_consequence','phys_health_consequence',
    'mental_health_interview','phys_health_interview','mental_vs_physical','obs_consequence'
]

for col in cols_yes_no:
    df[col] = (df[col] == 'yes').astype(int)


In [116]:
continent_map = {
    "North America": [
        "united states", "canada", "mexico", "bahamas, the", "costa rica"
    ],
    "South America": [
        "brazil", "colombia", "uruguay"
    ],
    "Europe": [
        "france", "united kingdom", "portugal", "netherlands", "switzerland",
        "poland", "germany", "slovenia", "austria", "ireland", "italy",
        "bulgaria", "sweden", "latvia", "romania", "belgium", "spain",
        "finland", "bosnia and herzegovina", "hungary", "croatia", "norway",
        "denmark", "greece", "moldova", "czech republic", "georgia"
    ],
    "Asia": [
        "india", "israel", "singapore", "japan", "thailand",
        "china", "philippines"
    ],
    "Eurasia": [
        "russia"
    ],
    "Africa": [
        "south africa", "zimbabwe", "nigeria"
    ],
    "Oceania": [
        "australia", "new zealand"
    ]
}

# inverte o dicionário para país → continente
country_to_continent = {
    country: cont
    for cont, countries in continent_map.items()
    for country in countries
}

df["continent"] = df["Country"].str.lower().map(country_to_continent)
df.drop(columns=['Country'], inplace=True)


In [118]:
male_terms = {
    "male", "m", "cis male", "male (cis)", "male ", "man", "cis man",
    "mal", "maile", "make", "msle", "mail", "malr"
}

df["gender"] = df["Gender"].str.strip().str.lower().apply(
    lambda x: 1 if x in male_terms else 0
)


In [119]:
non_cis_terms = {
    "trans-female", "trans woman", "female (trans)",
    "non-binary", "enby", "fluid", "genderqueer",
    "androgyne", "agender", "queer", "queer/she/they",
    "male-ish", "something kinda male?", 
    "male leaning androgynous", "guy (-ish) ^_^",
    "ostensibly male, unsure what that really means","nah",
    "all","neuter","a little about you","p"
}

df["non_cis"] = df["Gender"].apply(lambda x: 1 if x in non_cis_terms else 0)

In [120]:
df.drop(columns=['Gender'], inplace=True)

In [ ]:
import pandas as pd

cols_to_encode = ["no_employees", "coworkers", "supervisor", "continent","leave"]

# Cria as variáveis dummies (one-hot)
df_encoded = pd.get_dummies(df, columns=cols_to_encode, drop_first=False, dtype=int)

print("Shape antes:", df.shape)
print("Shape depois:", df_encoded.shape)

# Remove as colunas originais (caso ainda existam)
df_encoded = df_encoded.drop(columns=cols_to_encode, errors="ignore")


Shape antes: (1241, 24)
Shape depois: (1241, 49)


In [123]:
df = df_encoded

In [124]:
df.columns = (
    df.columns
    .str.strip()           # remove espaços extras no início/fim
    .str.lower()           # deixa tudo minúsculo
    .str.replace(" ", "_") # troca espaços por underscore
    .str.replace(".", "_")
    .str.replace("/", "_")
    .str.replace("-", "_")
)

In [126]:
base_path = Path("../data/processed")  
datapath = base_path / dataset_code  

df.to_csv(datapath / f"{dataset_code}_pp.csv", index=False)

# Heart Desease

In [9]:
df = pd.read_csv(base_path / f"4_heart_desease/heart_desease.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\UFRGS\\Mestrado\\fairnes_treatment\\data\\raw\\4_heart_desease\\heart_desease.csv'

In [15]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
df.rename(columns={"num": "target"}, inplace=True)
df['target'] = np.where(df['target'].isin([0]), 0, 1)

In [21]:
df.dropna(inplace=True)

In [28]:
df.to_csv(destination_path / f"4_heart_desease/4_heart_desease_pp.csv", index=False)

# Liver

In [3]:
df = pd.read_csv(base_path / f"3_liver/liver.csv")

In [7]:
df.isna().sum()

Age                           0
Gender                        0
Total_Bilirubin               0
Direct_Bilirubin              0
Alkaline_Phosphotase          0
Alamine_Aminotransferase      0
Aspartate_Aminotransferase    0
Total_Protiens                0
Albumin                       0
Albumin_and_Globulin_Ratio    4
Dataset                       0
dtype: int64

# Aids

In [17]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
aids_clinical_trials_group_study_175 = fetch_ucirepo(id=890) 
  
# data (as pandas dataframes) 
X = aids_clinical_trials_group_study_175.data.features 
y = aids_clinical_trials_group_study_175.data.targets 
  
# metadata 
print(aids_clinical_trials_group_study_175.metadata) 
  
# variable information 
print(aids_clinical_trials_group_study_175.variables) 


{'uci_id': 890, 'name': 'AIDS Clinical Trials Group Study 175', 'repository_url': 'https://archive.ics.uci.edu/dataset/890/aids+clinical+trials+group+study+175', 'data_url': 'https://archive.ics.uci.edu/static/public/890/data.csv', 'abstract': 'The AIDS Clinical Trials Group Study 175 Dataset contains healthcare statistics and categorical information about patients who have been diagnosed with AIDS. This dataset was initially published in 1996. The prediction task is to predict whether or not each patient died within a certain window of time or not. ', 'area': 'Health and Medicine', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Tabular', 'Multivariate'], 'num_instances': 2139, 'num_features': 23, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Sexual Orientation', 'Race', 'Gender'], 'target_col': ['cid'], 'index_col': ['pidnum'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1996, 'last_updated': 'Fri Nov 

In [4]:
df = pd.read_csv(base_path / f"6_aids/aids.csv")

In [5]:
df = df.drop(columns=['Unnamed: 0','pidnum','time'])



In [6]:
df.isna().sum()

cid        0
trt        0
age        0
wtkg       0
hemo       0
homo       0
drugs      0
karnof     0
oprior     0
z30        0
zprior     0
preanti    0
race       0
gender     0
str2       0
strat      0
symptom    0
treat      0
offtrt     0
cd40       0
cd420      0
cd80       0
cd820      0
dtype: int64

In [7]:
df = df.rename(columns={"cid": "target"})

In [8]:
len(df.columns)

23

In [9]:
df.to_csv(destination_path / f"6_aids/6_aids_pp.csv", index=False)

# Obesity

In [4]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
myocardial_infarction_complications = fetch_ucirepo(id=544) 
  
# data (as pandas dataframes) 
X = pd.DataFrame(myocardial_infarction_complications.data.features) 
y = pd.DataFrame(myocardial_infarction_complications.data.targets) 

In [5]:
# dataset_name = 'obesity'
# import_df = pd.concat([X, y], axis=1)
# import_df.to_csv(datapath + f"{dataset_name}.csv")

# metadata = pd.DataFrame(myocardial_infarction_complications.variables)
# metadata.to_csv(datapath + f"{dataset_name}_metadata.csv")

In [6]:
df = pd.read_csv(base_path / f"7_obesity/obesity.csv")

In [7]:
# Higienização dos nomes das colunas
df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()

# Verificando as colunas higienizadas
print(df.columns)

Index(['gender', 'age', 'height', 'weight', 'family_history_with_overweight',
       'favc', 'fcvc', 'ncp', 'caec', 'smoke', 'ch2o', 'scc', 'faf', 'tue',
       'calc', 'mtrans', 'nobeyesdad'],
      dtype='object')


In [8]:
df = df.rename(columns={"nobeyesdad": "target"})

In [9]:
df.isnull().sum()

gender                            0
age                               0
height                            0
weight                            0
family_history_with_overweight    0
favc                              0
fcvc                              0
ncp                               0
caec                              0
smoke                             0
ch2o                              0
scc                               0
faf                               0
tue                               0
calc                              0
mtrans                            0
target                            0
dtype: int64

In [10]:
df['target'].groupby(df['target']).size()

target
Insufficient_Weight    272
Normal_Weight          287
Obesity_Type_I         351
Obesity_Type_II        297
Obesity_Type_III       324
Overweight_Level_I     290
Overweight_Level_II    290
Name: target, dtype: int64

In [11]:
target_schemes = {
    # 1) Obeso (I, II, III) vs não-obeso
    "obese_I_II_III_vs_rest": [
        "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III"
    ],

    # 2) Obesidade severa (II, III) vs demais
    "severe_obesity_II_III_vs_rest": [
        "Obesity_Type_II", "Obesity_Type_III"
    ],

    # 3) Normal vs alterado
    "normal_vs_rest": [
        "Normal_Weight"
    ],

    # 4) Só Obesity_Type_II vs demais
    "obesity_type_II_vs_rest": [
        "Obesity_Type_II"
    ],

    # 5) Só Obesity_Type_III vs demais
    "obesity_type_III_vs_rest": [
        "Obesity_Type_III"
    ],

    # 6) Só Insufficient_Weight vs demais
    "insufficient_vs_rest": [
        "Insufficient_Weight"
    ],

    # 7) Só Overweight_Level_II vs demais
    "overweight_II_vs_rest": [
        "Overweight_Level_II"
    ],
}


In [12]:
def simple_fairness_test(df, sensitive_col, target_col, positive_classes):
    # cria target binário
    y_bin = df[target_col].isin(positive_classes).astype(int)

    # taxa de positivos por grupo
    rates = y_bin.groupby(df[sensitive_col]).mean()

    p_female = rates.get("Female", float("nan"))
    p_male = rates.get("Male", float("nan"))

    spd = p_female - p_male          # Statistical Parity Difference
    di = p_female / p_male if p_male > 0 else float("inf")  # Disparate Impact

    return p_female, p_male, spd, di


In [13]:
results = []

for name, positives in target_schemes.items():
    p_f, p_m, spd, di = simple_fairness_test(
        df, sensitive_col="gender", target_col="target",
        positive_classes=positives
    )
    results.append({
        "task": name,
        "positives": ", ".join(positives),
        "P(Y=1|Female)": p_f,
        "P(Y=1|Male)": p_m,
        "SPD (F - M)": spd,
        "DI (F / M)": di,
    })

fairness_screen = pd.DataFrame(results)
fairness_screen


,task,positives,P(Y=1|Female),P(Y=1|Male),SPD (F - M),DI (F / M)
0,obese_I_II_III_vs_rest,"Obesity_Type_I, Obesity_Type_II, Obesity_Type_III",0.461170,0.459738,0.001432,1.003115
1,severe_obesity_II_III_vs_rest,"Obesity_Type_II, Obesity_Type_III",0.311601,0.277154,0.034448,1.124291
2,normal_vs_rest,Normal_Weight,0.135187,0.136704,-0.001517,0.988902
3,obesity_type_II_vs_rest,Obesity_Type_II,0.001918,0.276217,-0.274300,0.006942
4,obesity_type_III_vs_rest,Obesity_Type_III,0.309684,0.000936,0.308747,330.742090
5,insufficient_vs_rest,Insufficient_Weight,0.165868,0.092697,0.073171,1.789361
6,overweight_II_vs_rest,Overweight_Level_II,0.098754,0.175094,-0.076340,0.564004


In [14]:
df['target'] = np.where(df['target'].isin(['Obesity_Type_II', 'Obesity_Type_III']), 1, 0)

In [15]:
df['calc'] = df['calc'].replace({'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always':3})
df['calc'] = df['calc'].astype(int)

df['caec'] = df['caec'].replace({'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always':3})
df['caec'] = df['caec'].astype(int)

C:\Users\gabri\AppData\Local\Temp\ipykernel_9272\3283504497.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['calc'] = df['calc'].replace({'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always':3})
C:\Users\gabri\AppData\Local\Temp\ipykernel_9272\3283504497.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['caec'] = df['caec'].replace({'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always':3})


In [16]:
df['family_history_with_overweight'] = df['family_history_with_overweight'].replace({'no': '0', 'yes': '1'})
df['family_history_with_overweight'] = df['family_history_with_overweight'].astype(int)

df['scc'] = df['scc'].replace({'no': '0', 'yes': '1'})
df['scc'] = df['scc'].astype(int)

df['favc'] = df['favc'].replace({'no': '0', 'yes': '1'})
df['favc'] = df['favc'].astype(int)

df['smoke'] = df['smoke'].replace({'no': '0', 'yes': '1'})
df['smoke'] = df['smoke'].astype(int)

In [17]:
df['gender'] = df['gender'].replace({'Female': '0', 'Male': '1'})
df['gender'] = df['gender'].astype(int)

In [18]:
df.mtrans.unique()

array(['Public_Transportation', 'Walking', 'Automobile', 'Motorbike',
       'Bike'], dtype=object)

In [19]:
# Generate dummies
dummies = pd.get_dummies(df['mtrans'], prefix='mtrans')

# Convert dummies to int type
dummies = dummies.astype(int)

dummies.columns = dummies.columns.str.lower()

# Concatenate with the original DataFrame
df = pd.concat([df, dummies], axis=1)

# Drop the 'mtrans' column from the DataFrame
df = df.drop(columns=['mtrans'])

In [ ]:
df.to_csv(destination_path / f"7_obesity/7_obesity_pp.csv", index=False)

: 

# Diabetes 

In [1]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
early_stage_diabetes_risk_prediction = fetch_ucirepo(id=529) 
  
# data (as pandas dataframes) 
X = early_stage_diabetes_risk_prediction.data.features 
y = early_stage_diabetes_risk_prediction.data.targets 
  
# metadata 
print(early_stage_diabetes_risk_prediction.metadata) 
  
# variable information 
print(early_stage_diabetes_risk_prediction.variables) 


{'uci_id': 529, 'name': 'Early Stage Diabetes Risk Prediction', 'repository_url': 'https://archive.ics.uci.edu/dataset/529/early+stage+diabetes+risk+prediction+dataset', 'data_url': 'https://archive.ics.uci.edu/static/public/529/data.csv', 'abstract': 'This dataset contains the sign and symptpom data of newly diabetic or would be diabetic patient. ', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 520, 'num_features': 16, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Age', 'Gender'], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2020, 'last_updated': 'Mon Mar 04 2024', 'dataset_doi': '10.24432/C5VG8H', 'creators': [], 'intro_paper': {'ID': 397, 'type': 'NATIVE', 'title': 'Likelihood Prediction of Diabetes at Early Stage Using Data Mining Techniques', 'authors': 'M. M. F. Islam, Rahatara Ferdousi, Sadikur Rahman, Humayra Yas

In [6]:
df = pd.concat([X, y], axis=1)

In [8]:
df.to_csv(base_path / f"16_diabetes/16_diabetes.csv", index=False)

In [14]:
mapping = {
    'Yes': 1, 'No': 0,
    'Positive': 1, 'Negative': 0,
    'Male': 1, 'Female': 0
}

cols_to_convert = [c for c in df.columns if c != 'age']

df[cols_to_convert] = df[cols_to_convert].replace(mapping).astype(int)

C:\Users\gabri\AppData\Local\Temp\ipykernel_15180\4294694992.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[cols_to_convert] = df[cols_to_convert].replace(mapping).astype(int)


In [15]:

df.rename(columns={"class": "target"}, inplace=True)

In [17]:
df.isna().sum()

age                   0
gender                0
polyuria              0
polydipsia            0
sudden_weight_loss    0
weakness              0
polyphagia            0
genital_thrush        0
visual_blurring       0
itching               0
irritability          0
delayed_healing       0
partial_paresis       0
muscle_stiffness      0
alopecia              0
obesity               0
target                0
dtype: int64

In [18]:
df.to_csv(destination_path / f"16_diabetes/16_diabetes_pp.csv", index=False)